# Losses: Regression, Classification, and Barron Adaptive Robust (C03 Topic 07)

**Citations:** Barron (2019 General Adaptive Robust Loss); Huber (1964 Robust Estimation); Lin et al. (2017 Focal Loss); Leng et al. (2022 PolyLoss); Ridnik et al. (2021 Asymmetric Loss ASL).

### Purpose & Topic Bridges
The objective function shapes gradient dynamics, outlier tolerance, and class imbalance resilience. This notebook implements Barron's General Adaptive Robust Loss, Huber Smooth L1 Loss, Multi-Class Focal Loss, Poly-1 Loss, and Asymmetric Loss (ASL) for extreme multi-label classification.

## 2. Mathematical Foundations & Governing Equations

1. **Barron Robust Loss:** $\rho(r, \alpha, c) = \frac{|\alpha - 2|}{\alpha} \left( \left( \frac{(r/c)^2}{|\alpha - 2|} + 1 \right)^{\alpha/2} - 1 \right)$, smoothly interpolating L2 ($\alpha=2$), Charbonnier ($\alpha=1$), Cauchy ($\alpha=0$), and Welsch ($\alpha \to -\infty$).
2. **Huber Loss:** $\mathcal{L}(r, \delta) = \frac{1}{2} r^2$ if $|r| \le \delta$, else $\delta (|r| - \frac{1}{2}\delta)$, providing $C^1$ smooth piecewise robust regression.
3. **Focal Loss:** $\mathcal{L}(p_t) = -(1 - p_t)^\gamma \log(p_t)$, down-weighting well-classified easy samples.
4. **PolyLoss (Poly-1):** $\mathcal{L} = -\log(p_t) + \epsilon_1 (1 - p_t)$, fine-tuning the leading Taylor polynomial term.
5. **Asymmetric Loss (ASL):** Decoupled focusing $\gamma_+ \le \gamma_-$ with shifted probability margin $p_m = \max(p - m, 0)$ to zero out easy background negatives.

In [ ]:
import sys, math
from pathlib import Path
import torch, torch.nn as nn, torch.nn.functional as F

repo_root = Path.cwd().resolve()
while not (repo_root / "core").exists() and repo_root.parent != repo_root: repo_root = repo_root.parent
if str(repo_root) not in sys.path: sys.path.insert(0, str(repo_root))
import core
core.setup_env(seed=42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ckpt_path = repo_root / "weights/03_building_blocks_and_losses/07_losses_regression_classification_and_barron_robust.safetensors"
weights, test_in, test_out, meta = core.parity.load_reference_checkpoint(ckpt_path, device=device)
print(f"Device: {device} | Checkpoint: {ckpt_path.name} | Loaded {len(weights)} tensors")

In [ ]:
class LossesUnifiedModule(nn.Module):
    def __init__(self, in_dim=8, out_dim=4):
        super().__init__()
        self.linear = nn.Linear(in_dim, out_dim)

    def forward(self, x): return self.linear(x)

    def barron_loss(self, r, alpha, c):
        d_a = abs(alpha - 2.0)
        scaled_sq = (r / c).pow(2)
        return ((d_a / alpha) * ((scaled_sq / d_a + 1.0).pow(alpha / 2.0) - 1.0)).mean()

    def huber_loss(self, r, delta):
        abs_r = torch.abs(r)
        return torch.where(abs_r <= delta, 0.5 * r.pow(2), delta * (abs_r - 0.5 * delta)).mean()

    def focal_loss(self, logits, targets, gamma):
        log_p = F.log_softmax(logits, dim=-1)
        p = torch.exp(log_p)
        b = logits.shape[0]
        t_log_p = log_p[torch.arange(b), targets]
        t_p = p[torch.arange(b), targets]
        return (-(1.0 - t_p).pow(gamma) * t_log_p).mean()

    def polyloss(self, logits, targets, eps_1):
        log_p = F.log_softmax(logits, dim=-1)
        p = torch.exp(log_p)
        b = logits.shape[0]
        t_log_p = log_p[torch.arange(b), targets]
        t_p = p[torch.arange(b), targets]
        return (-t_log_p + eps_1 * (1.0 - t_p)).mean()

    def asymmetric_loss(self, logits, targets, gp, gn, margin, eps=1e-8):
        p = torch.sigmoid(logits)
        p_m = torch.clamp(p - margin, min=0.0)
        l_pos = -targets * (1.0 - p).pow(gp) * torch.log(p.clamp(min=eps))
        l_neg = -(1.0 - targets) * p_m.pow(gn) * torch.log((1.0 - p_m).clamp(min=eps))
        return (l_pos + l_neg).mean()

model = LossesUnifiedModule().to(device)
with torch.no_grad():
    model.linear.weight.copy_(weights['weight'].t()); model.linear.bias.copy_(weights['bias'])
print('LossesUnifiedModule initialized successfully.')

In [ ]:
r_b, a_b, c_b = weights['r_barron'], float(weights['alpha_barron'][0]), float(weights['c_barron'][0])
r_h, d_h = weights['r_huber'], float(weights['delta_huber'][0])
l_clf, t_clf = weights['logits_clf'], weights['targets_clf'].to(torch.int64)
g_f, e_p = float(weights['gamma_focal'][0]), float(weights['eps_poly'][0])
l_asl, t_asl = weights['logits_asl'], weights['targets_asl']
gp, gn, m_asl = float(weights['gamma_pos'][0]), float(weights['gamma_neg'][0]), float(weights['margin_asl'][0])

err_b = (model.barron_loss(r_b, a_b, c_b) - weights['loss_barron'][0]).abs().item()
assert err_b < 1e-5, f'Barron parity error: {err_b:.2e}'

err_h = (model.huber_loss(r_h, d_h) - weights['loss_huber'][0]).abs().item()
assert err_h < 1e-5, f'Huber parity error: {err_h:.2e}'

err_f = (model.focal_loss(l_clf, t_clf, g_f) - weights['loss_focal'][0]).abs().item()
assert err_f < 1e-5, f'Focal parity error: {err_f:.2e}'

err_p = (model.polyloss(l_clf, t_clf, e_p) - weights['loss_poly'][0]).abs().item()
assert err_p < 1e-5, f'PolyLoss parity error: {err_p:.2e}'

err_asl = (model.asymmetric_loss(l_asl, t_asl, gp, gn, m_asl) - weights['loss_asl'][0]).abs().item()
assert err_asl < 1e-5, f'ASL parity error: {err_asl:.2e}'

err_anchor = (model(test_in) - test_out).abs().max().item()
assert err_anchor < 1e-5, f'Anchor parity error: {err_anchor:.2e}'

print(f'All 6 invariants verified! Barron: {err_b:.2e}, Huber: {err_h:.2e}, Focal: {err_f:.2e}, Poly: {err_p:.2e}, ASL: {err_asl:.2e}, Anchor: {err_anchor:.2e}')

In [ ]:
track_fns = {
    'Linear Anchor': lambda: model(test_in),
    'Barron Robust Loss': lambda: model.barron_loss(r_b, a_b, c_b),
    'Huber Loss': lambda: model.huber_loss(r_h, d_h),
    'Focal Loss': lambda: model.focal_loss(l_clf, t_clf, g_f),
    'Poly-1 Loss': lambda: model.polyloss(l_clf, t_clf, e_p),
    'Asymmetric Loss (ASL)': lambda: model.asymmetric_loss(l_asl, t_asl, gp, gn, m_asl),
}
bench_results = core.benchmark.compare_tracks(track_fns, warmup=3, iters=15, batch_size=4)
print(core.benchmark.format_benchmark_table(bench_results))

In [ ]:
# Pathological Failure Mode: Unnormalized Barron Parameter Optimization Collapse vs Fixed / Normalized Estimators
# If alpha is treated as an unconstrained learnable parameter without partition function log Z(alpha),
# gradient descent drives alpha -> -inf because the raw loss value trivially collapses to a bounded constant
residuals = torch.tensor([5.0, 10.0, 15.0], device=device)
loss_l2 = model.barron_loss(residuals, alpha=1.999, c=1.0) # alpha~2 (L2 standard limit)
loss_cauchy = model.barron_loss(residuals, alpha=0.001, c=1.0) # alpha~0 (Cauchy robust limit)
loss_collapsed = model.barron_loss(residuals, alpha=-20.0, c=1.0) # alpha -> -inf collapse

print(f'[LOSS VALUES] L2 (alpha~2): {loss_l2.item():.2f} | Cauchy (alpha~0): {loss_cauchy.item():.2f} | Collapsed (alpha=-20): {loss_collapsed.item():.2f}')
assert loss_collapsed < loss_cauchy < loss_l2, 'Expected descending loss magnitude as alpha decreases'
assert loss_collapsed.item() <= 1.15, f'Expected unnormalized Barron to collapse near 1.0, got {loss_collapsed.item():.2f}'
print('[FAILURE DEMONSTRATED] Unnormalized alpha optimization trivially collapses loss without fitting.')

# Fixed shape parameter (alpha=1.0 Charbonnier) maintains healthy non-zero gradient force on residuals
r_test = torch.tensor([5.0], requires_grad=True, device=device)
loss_stable = model.barron_loss(r_test, alpha=1.0, c=1.0)
loss_stable.backward()
grad_val = r_test.grad.abs().item()
print(f'[STABILIZED] Fixed alpha=1.0 maintains bounded, non-vanishing gradient force: {grad_val:.4f}')
assert 0.95 < grad_val < 1.05, f'Expected saturated gradient ~1.0, got {grad_val:.4f}'
assert torch.allclose(model(test_in), test_out, atol=1e-5), 'Linear anchor mismatch'
print('[CORRECT] Robust losses and linear anchor verified successfully.')

## 8. Algorithmic Takeaways & Cross-Domain Isomorphisms

1. **Continuous Robust Interpolation:** Barron's adaptive loss unifies classical estimators under a single continuous manifold $(\alpha, c)$. Learning $\alpha$ requires continuous density normalization $\log Z(\alpha)$.
2. **Dynamic Focal Suppression:** Modulating cross-entropy by $(1 - p_t)^\gamma$ breaks the gradient bottleneck in extreme class imbalance, enabling one-stage dense detection without hard-negative mining.
3. **Asymmetric Discarding:** In multi-label recognition with 10k+ classes, ASL's probability margin $p_m = \max(p - m, 0)$ completely eliminates easy negative gradients, preventing gradient drowning.

---
**Curriculum Navigation:** [Master Index](file:///c:/Users/Liam/Code/AI-Examples/README.md) | [Previous: 06_regularization_and_training_stabilizers](file:///c:/Users/Liam/Code/AI-Examples/notebooks/03_building_blocks_and_losses/06_regularization_and_training_stabilizers.ipynb)